# Global LULUCF + vegetation zonal stats by IPCC land use

This notebook reads global 10×10 tile-level zonal stats parquet files one at a time and summarizes them.

Outputs use sums only:

- `value` = sum of flux values
- `area_ha` = sum of area

The workflow is:
1. Read one tile parquet.
2. Summarize by strata.
3. Write compact tile summary to disk.
4. Delete raw tile data from memory.
5. After all tile summaries exist, combine compact summaries into one global master table.
6. Export Excel, CSV, and parquet.

In [ ]:
from pathlib import Path
import gc
import time

import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
# Update these paths if needed.
input_dir = Path(r"/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global")
output_dir = input_dir
file_keyword = "global"

tile_summary_dir = output_dir / f"tile_summaries__{file_keyword}"
tile_summary_dir.mkdir(parents=True, exist_ok=True)

print("Input folder:", input_dir)
print("Input exists:", input_dir.exists())
print("Output folder:", output_dir)
print("Tile summary folder:", tile_summary_dir)

In [ ]:
numeric_to_ipcc_class = {
    0: "Unassigned",
    1: "Settlements",
    2: "Cropland",
    3: "Forest Land",
    4: "Grassland",
    5: "Wetlands",
    6: "Other Land (Bare)",
    7: "Other Land (Water)",
    8: "Other Land (Snow/Ice)",
}

assignment_rules = [
    {
        "name": "crop_to_cropland",
        "match": {"land_state_broad_class": "crop"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "short_veg_to_grassland",
        "match": {"land_state_broad_class": "short_veg"},
        "fill": {"IPCC_summary": 44, "IPCC_change": 44, "IPCC_class": 4, "IPCC_class_name": "Grassland"},
    },
    {
        "name": "mangrove_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "mangrove"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "natural_tree_cover_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "natural_tree_cover"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "oil_palm_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "oil_palm"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "non_oil_palm_planted_trees_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "non_oil_palm_planted_trees"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "trees_in_other_land_covers_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "trees_in_other_land_covers"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "no_flux_stays_unassigned",
        "match": {"land_state_broad_class": "no_flux"},
        "fill": {"IPCC_summary": 0, "IPCC_change": 0, "IPCC_class": 0, "IPCC_class_name": "Unassigned"},
    },
]

In [ ]:
def log(message):
    print(f"{time.strftime('%Y%m%d_%H_%M_%S')}: {message}", flush=True)


def find_tile_parquets(input_dir, file_keyword):
    return sorted(
        p for p in input_dir.glob("*.parquet")
        if "ipcc_lulucf_zonal_stats_" in p.name
        and file_keyword in p.name
        and "_wide_" not in p.name
        and "summary" not in p.name
        and "combined" not in p.name
    )


def tile_id_from_parquet_name(parquet_file):
    parts = parquet_file.stem.split("_")
    if len(parts) >= 6:
        return f"{parts[4]}_{parts[5]}"
    return parquet_file.stem


def coalesce_node_code_column(df):
    if "IPCC_node_code" in df.columns:
        return df

    if "IPCC_node" in df.columns:
        return df.rename(columns={"IPCC_node": "IPCC_node_code"})

    return df


def coalesce_primary_forest_column(df):
    # Keep the requested grouping column name stable even if the source has a slightly different name.
    possible_cols = [
        "starting_composite_primary_forest",
        "starting_composite_primary_forest_2001",
        "composite_primary_forest",
    ]

    if "starting_composite_primary_forest" in df.columns:
        return df

    for col in possible_cols:
        if col in df.columns:
            return df.rename(columns={col: "starting_composite_primary_forest"})

    # If the source does not contain the column, keep a sentinel so the notebook still runs.
    df["starting_composite_primary_forest"] = "Unassigned"
    return df


def add_flux_source(df):
    df["flux_source"] = df["analysis_layer"].apply(
        lambda x: "LULUCF" if str(x).startswith("LULUCF_") else "vegetation"
    )
    return df


LAYERS_TO_KEEP = [
    # LULUCF layers
    "LULUCF_gross_emissions__all_C_pools__all_gases__MgCO2e",
    "LULUCF_gross_removals__all_C_pools__MgCO2",
    "LULUCF_net_flux__all_C_pools__all_gases__MgCO2e",

    # Vegetation layers, if present
    "gross_emissions__all_C_pools__all_gases__MgCO2e",
    "gross_removals__all_C_pools__MgCO2",
    "net_flux__all_C_pools__all_gases__MgCO2e",
]


SUMMARY_COLS = [
    "flux_source",
    "IPCC_summary",
    "IPCC_node_code",
    "year",
    "IPCC_change",
    "IPCC_class",
    "IPCC_class_name",
    "starting_composite_primary_forest",
    "land_state_broad_class",
    "tall_veg_type",
    "land_state_detailed_class",
    "land_state_meaning",
    "analysis_layer",
]

VALUE_COLS = ["value", "area_ha"]

OUTPUT_COLS = SUMMARY_COLS + VALUE_COLS


def apply_assignment_rule(df, rules=assignment_rules):
    if "IPCC_class_name" not in df.columns:
        return df

    df["IPCC_class_name"] = df["IPCC_class_name"].fillna("Unassigned")
    base_unassigned = df["IPCC_class_name"].astype(str).eq("Unassigned")

    for rule in rules:
        mask = base_unassigned.copy()

        for col, val in rule["match"].items():
            if col not in df.columns:
                mask &= False
            else:
                mask &= df[col].astype(str).eq(str(val))

        if mask.any():
            for fill_col, fill_val in rule["fill"].items():
                df.loc[mask, fill_col] = fill_val

    for col in ["IPCC_summary", "IPCC_change", "IPCC_class"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

    class_map_mask = df["IPCC_class_name"].isna() | df["IPCC_class_name"].astype(str).eq("")
    if "IPCC_class" in df.columns:
        df.loc[class_map_mask, "IPCC_class_name"] = df.loc[class_map_mask, "IPCC_class"].map(numeric_to_ipcc_class).fillna("Unassigned")

    return df

In [ ]:
def summarize_one_tile(parquet_file, tile_summary_dir):
    tile_id = tile_id_from_parquet_name(parquet_file)
    out_path = tile_summary_dir / f"{tile_id}__summary.parquet"

    if out_path.exists():
        log(f"Skipping {tile_id}; tile summary already exists")
        return out_path

    log(f"Reading {tile_id}: {parquet_file.name}")

    df = pd.read_parquet(parquet_file)
    df = coalesce_node_code_column(df)
    df = coalesce_primary_forest_column(df)

    if "analysis_layer" not in df.columns:
        raise ValueError(f"analysis_layer missing from {parquet_file.name}")

    df = add_flux_source(df)
    df = df[df["analysis_layer"].isin(LAYERS_TO_KEEP)].copy()

    if df.empty:
        log(f"No matching flux layers found in {tile_id}. Writing empty summary.")
        empty = pd.DataFrame(columns=OUTPUT_COLS)
        empty.to_parquet(out_path, index=False)
        del df, empty
        gc.collect()
        return out_path

    df = apply_assignment_rule(df)

    missing_cols = [c for c in SUMMARY_COLS + VALUE_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing expected columns in {parquet_file.name}: {missing_cols}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    summary = (
        df.groupby(SUMMARY_COLS, dropna=False, as_index=False)
          .agg({
              "value": "sum",
              "area_ha": "sum",
          })
    )

    summary = summary[OUTPUT_COLS].sort_values(SUMMARY_COLS).reset_index(drop=True)
    summary.to_parquet(out_path, index=False)

    log(f"Wrote {tile_id} summary: {len(summary):,} rows -> {out_path.name}")

    del df, summary
    gc.collect()

    return out_path


def combine_tile_summaries(tile_summary_dir):
    summary_files = sorted(tile_summary_dir.glob("*__summary.parquet"))

    if not summary_files:
        raise RuntimeError(f"No tile summary files found in {tile_summary_dir}")

    master = None

    for idx, summary_file in enumerate(summary_files, start=1):
        part = pd.read_parquet(summary_file)

        if master is None:
            master = part
        else:
            master = pd.concat([master, part], ignore_index=True)
            del part
            gc.collect()

        master = (
            master.groupby(SUMMARY_COLS, dropna=False, as_index=False)
                  .agg({
                      "value": "sum",
                      "area_ha": "sum",
                  })
        )

        if idx % 25 == 0 or idx == len(summary_files):
            log(f"Combined {idx:,}/{len(summary_files):,}; master rows={len(master):,}")

    master = master[OUTPUT_COLS].sort_values(SUMMARY_COLS).reset_index(drop=True)
    return master


def export_summary(master_df, output_dir, output_stem=None):
    timestamp = time.strftime("%Y%m%d_%H_%M_%S")

    if output_stem is None:
        output_stem = f"global_summary__{timestamp}"

    excel_out = output_dir / f"{output_stem}.xlsx"
    csv_out = output_dir / f"{output_stem}.csv"
    parquet_out = output_dir / f"{output_stem}.parquet"

    master_df.to_csv(csv_out, index=False)
    master_df.to_parquet(parquet_out, index=False)

    with pd.ExcelWriter(excel_out, engine="xlsxwriter") as writer:
        sheet_name = "LULUCF_flux_master"
        master_df.to_excel(writer, sheet_name=sheet_name, index=False)

        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        header_format = workbook.add_format({
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "align": "center",
            "valign": "vcenter",
        })
        numeric_format = workbook.add_format({"num_format": "#,##0.000"})
        integer_format = workbook.add_format({"num_format": "#,##0"})
        text_format = workbook.add_format({"text_wrap": False})

        for col_idx, col_name in enumerate(master_df.columns):
            worksheet.write(0, col_idx, col_name, header_format)

            if col_name in ["value", "area_ha"]:
                worksheet.set_column(col_idx, col_idx, 18, numeric_format)
            elif col_name in ["IPCC_summary", "IPCC_node_code", "year", "IPCC_change", "IPCC_class"]:
                worksheet.set_column(col_idx, col_idx, 14, integer_format)
            else:
                width = min(max(len(str(col_name)) + 4, 16), 55)
                worksheet.set_column(col_idx, col_idx, width, text_format)

        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(0, 0, max(len(master_df), 1), len(master_df.columns) - 1)

    return excel_out, csv_out, parquet_out

## Find tile parquet files

In [ ]:
parquet_files = find_tile_parquets(input_dir, file_keyword)

print("Tile parquet count:", len(parquet_files))
parquet_files[:5], parquet_files[-5:]

## Optional: inspect one tile before running all

This is useful to verify that `starting_composite_primary_forest` exists in the raw parquet.

In [ ]:
if parquet_files:
    sample = pd.read_parquet(parquet_files[0])
    print(sample.shape)
    print(sample.columns.tolist())
    print("analysis_layer values:")
    print(sorted(sample["analysis_layer"].dropna().unique().tolist()))
    del sample
    gc.collect()

## Phase 1: summarize each tile


In [ ]:
for idx, parquet_file in enumerate(parquet_files, start=1):
    log(f"Tile {idx:,}/{len(parquet_files):,}")
    summarize_one_tile(parquet_file, tile_summary_dir)
    gc.collect()

## Phase 2: combine compact tile summaries

In [ ]:
master_df = combine_tile_summaries(tile_summary_dir)

print(master_df.shape)
master_df.head(25)

## QA

In [ ]:
qa = (
    master_df.groupby(["flux_source", "analysis_layer"], dropna=False)[["value", "area_ha"]]
             .sum()
             .reset_index()
             .sort_values(["flux_source", "analysis_layer"])
)

qa

In [ ]:
master_df.groupby(["IPCC_class", "IPCC_class_name"], dropna=False)[["value", "area_ha"]].sum().reset_index()

## Phase 3: export Excel, CSV, and parquet

In [ ]:
excel_out, csv_out, parquet_out = export_summary(master_df, output_dir)

print("Excel:", excel_out)
print("CSV:", csv_out)
print("Parquet:", parquet_out)

## Optional QA checks

In [ ]:
print("Rows:", len(master_df))
print("Total value:", master_df["value"].sum())
print("Total area_ha:", master_df["area_ha"].sum())

In [ ]:
master_df.groupby("year", dropna=False)[["value", "area_ha"]].sum()